# Pipeline

Reopen the MiniLM store from Day 2. Retrieve chunks and answer with Azure `gpt-4o-mini` at temperature 0.


## Setup


In [1]:
import sys
from pathlib import Path

from dotenv import load_dotenv

cwd = Path.cwd().resolve()
repo_root = next((p for p in [cwd, *cwd.parents] if (p / ".git").exists()), cwd)
load_dotenv(repo_root / ".env", override=True)
sys.path.insert(0, str(repo_root / "W5-RAG"))

from langchain_core.messages import HumanMessage, SystemMessage
from langchain_huggingface import HuggingFaceEmbeddings

from store import CHAT_MODEL, MINILM, azure_chat, client, open_store

print(CHAT_MODEL)


gpt-4o-mini


## Store


In [2]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = open_store(MINILM, embeddings)
print(client().get_collection(MINILM).count(), "vectors")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

413 vectors


## Temperature

Temperature chooses the next token. `0` always takes the most likely token. It does not add creativity, and it does not make the answer reproducible. Put variety in the system prompt.


## Retriever and model


In [3]:
retriever = vectorstore.as_retriever()
llm = azure_chat(temperature=0)

docs = retriever.invoke("Who is Avery?")
print(len(docs))
print(docs[0].page_content[:300])


4
## Other HR Notes
- **Professional Development**: Avery has actively participated in leadership training programs and industry conferences, representing Insurellm and fostering partnerships.  
- **Diversity & Inclusion Initiatives**: Avery has championed a commitment to diversity in hiring practices


In [4]:
print(llm.invoke("Who is Avery?").content)


The name "Avery" can refer to various individuals, characters, or concepts depending on the context. It could be a first name or surname, and it is used by many people across different fields, including entertainment, sports, literature, and more. If you have a specific Avery in mind—such as a celebrity, fictional character, or historical figure—please provide more details, and I can give you more information!


## Answer


In [5]:
SYSTEM_PROMPT = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""


def answer_question(question, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    response = llm.invoke([
        SystemMessage(content=SYSTEM_PROMPT.format(context=context)),
        HumanMessage(content=question),
    ])
    return response.content


print(answer_question("Who is Avery Lancaster?", []))


Avery Lancaster is the Co-Founder and Chief Executive Officer (CEO) of Insurellm, a leading insurance technology provider. She co-founded the company in 2015 and has played a crucial role in guiding its growth and positioning in the insurance market. Avery is known for her innovative leadership strategies and expertise in risk management. She has actively participated in professional development, diversity and inclusion initiatives, and community engagement efforts, particularly focusing on financial literacy programs for underserved populations. Under her leadership, Insurellm has achieved significant milestones and recognition in the industry.


## Azure embedding store

`text-embedding-3-large`, chunks of 500 characters with overlap 200. `app.py` reads this store and shows the retrieved context beside the answer.


In [6]:
from ingest import build

build()


Loaded 76 documents
Divided into 970 chunks


970 vectors, 3072 dimensions, collection large


In [7]:
from answer import answer_question as answer_with_azure

text, docs = answer_with_azure("Who is Lancaster?")
print(text)
print(len(docs), "chunks")
print(docs[0].metadata.get("source", ""))


Avery Lancaster is the Co-Founder and Chief Executive Officer (CEO) of Insurellm, an insurance tech startup founded in 2015. She has played a crucial role in positioning the company as a key player in the insurance technology landscape, demonstrating resilience and adaptability throughout her career. Avery is known for her innovative leadership strategies and risk management expertise.
10 chunks
/home/akalanka/SSD/Akalanka/cources/udemy-agentic-ai-core-track/W5-RAG/knowledge-base/employees/Avery Lancaster.md


## Page


In [8]:
from app import launch

launch()


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
